# Pencarian *random search* di Kaggle (`kaggle_pencarian.ipynb`)
Program pembantu untuk Tahap 11 (F10): RS 5 *run* × 40 kandidat di **satu T4**, dengan lingkungan dan pustaka yang **sama persis** dengan Colab. Hasilnya diimpor di Colab (Tahap 11.2). Data uji tidak ada di paket (F14).

## Cara menjalankan
1. Panel kanan *Settings*: *Accelerator* **GPU T4 x2** (program hanya memakai GPU 0) dan **Internet On**.
2. Jalankan **K0 saja** sebagai audit (sesi baru, belum ada sel lain yang dijalankan). Kirim output-nya.
3. *Add Input* → dataset paket (`paket_kaggle.zip`, Private), lalu jalankan **K1–K2** secara interaktif. Bila K2 lulus, hentikan sesi (*Stop session*).
4. *Save Version* → **Save & Run All (Commit)**. K0–K4 berjalan tanpa interaksi; K3 berhenti rapi setelah 11 jam.
5. Bila belum selesai: *Add Input* → *Notebook Output Files* (versi sebelumnya), lalu *commit* lagi; *state* dilanjutkan otomatis.
6. Unduh `keluaran_kaggle_*.zip` dari tab *Output*, unggah ke `MyDrive/skripsi_wondr_v4/kaggle/`, lalu jalankan Tahap 11.2 di Colab.

Bila RS selesai lebih dulu, `TUGAS` di K3 boleh ditambah *run* PSO mulai dari *seed* terbesar: `("pso", 15), ("pso", 14), …`.

## K0 — Lingkungan
Sebelum torch di-*import*: `CUDA_VISIBLE_DEVICES=0`, determinisme, dan senyap. Menampilkan `nvidia-smi`, Python, dan versi bawaan; berhenti bila *driver* tidak mendukung CUDA di `VERSI`; memasang `VERSI`; cek satu GPU T4 dan versi = `VERSI`. **Bisa dijalankan sendiri tanpa paket (audit).**

In [ ]:
# ══ K0 — Lingkungan Kaggle (satu T4, VERSI terkunci; bisa dijalankan sendiri sebagai audit tanpa paket) ══
# Package: os, re, sys, json, time, shutil, platform, subprocess, importlib, pathlib (bawaan Python); torch (setelah VERSI)
import os, re, sys, json, time, shutil, platform, subprocess, importlib.util, importlib.metadata as im
from pathlib import Path
T_MULAI = time.time()                                        # batas waktu K3 dihitung dari awal K0 (F10)

# 1) Sebelum torch di-import: hanya GPU 0 (F11), determinisme cuBLAS, dan senyap (CLAUDE.md 3.3-3.4)
if "torch" in sys.modules:
    raise RuntimeError("torch sudah di-import sebelum K0. Restart sesi (Run → Restart & clear cell outputs), lalu jalankan K0 paling awal.")
os.environ.update({"CUDA_VISIBLE_DEVICES": "0", "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "TOKENIZERS_PARALLELISM": "false",
                   "TRANSFORMERS_VERBOSITY": "error", "TRANSFORMERS_NO_ADVISORY_WARNINGS": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                   "TQDM_DISABLE": "1", "USE_TF": "0", "TF_CPP_MIN_LOG_LEVEL": "3"})

# 2) Lingkungan terkunci (F7). Tiga konstanta ini SAMA PERSIS dengan Tahap 0.1 skripsi_wondr_v4_colab.ipynb.
VERSI = {"torch": "2.14.1+cu130", "transformers": "5.22.1", "tokenizers": "0.23.3", "huggingface-hub": "1.33.0",
         "numpy": "2.4.6", "pandas": "2.2.3", "scikit-learn": "1.6.1", "scipy": "1.16.3", "pyarrow": "23.0.1"}
INDEKS_TORCH = "https://download.pytorch.org/whl/cu130"          # build CUDA 13.0 untuk torch di VERSI
REVISI = "c2cd0b51ddce6580eb35263b39b0a1e5fb0a39e2"              # commit indobenchmark/indobert-base-p1 (audit 13-11-2026)

# 3) nvidia-smi, Python, dan versi bawaan Kaggle (sebelum pemasangan)
def terpasang(p):                                 # versi terpasang tanpa meng-import; torch dari torch/version.py (+cuXXX)
    try:
        if p == "torch" and importlib.util.find_spec("torch"):
            f = Path(next(iter(importlib.util.find_spec("torch").submodule_search_locations))) / "version.py"
            return re.search(r"__version__\s*=\s*['\"]([^'\"]+)", f.read_text()).group(1)
        return im.version(p)
    except (im.PackageNotFoundError, OSError, AttributeError):
        return None
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else ""
cuda_drv = (re.findall(r"CUDA Version:\s*([\d.]+)", smi) or ["0.0"])[0]
cuda_perlu = VERSI["torch"].split("+cu")[1]
cuda_perlu = (int(cuda_perlu[:-1]), int(cuda_perlu[-1]))     # "130" -> (13, 0)
bawaan = {p: terpasang(p) for p in VERSI}
print(f"══════════ K0 — LINGKUNGAN KAGGLE ══════════\n{smi.strip() or 'nvidia-smi tidak ada'}\n{'─' * 48}")
print(f"Python {platform.python_version()} · driver CUDA {cuda_drv} · build torch di VERSI butuh CUDA {cuda_perlu[0]}.{cuda_perlu[1]}")
if tuple(map(int, cuda_drv.split(".")[:2])) < cuda_perlu:
    raise RuntimeError(f"Driver Kaggle hanya mendukung CUDA {cuda_drv}; VERSI butuh CUDA {cuda_perlu[0]}.{cuda_perlu[1]}. "
                       "Berhenti: laporkan output ini (VERSI perlu ditetapkan ulang untuk kedua platform).")

# 4) Pasang versi yang berbeda dari VERSI (torch belum di-import, jadi tidak perlu restart)
beda = {p: v for p, v in bawaan.items() if v != VERSI[p]}
if beda:
    perintah = ([[sys.executable, "-m", "pip", "install", "-q", f"torch=={VERSI['torch']}", "--index-url", INDEKS_TORCH]] if "torch" in beda else []) \
             + ([[sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={VERSI[p]}" for p in beda if p != "torch"]]] if set(beda) - {"torch"} else [])
    for c in perintah:
        r = subprocess.run(c, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("pip install gagal:\n" + r.stderr[-2000:])
MODUL = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn", "pyarrow": "pyarrow",
         "transformers": "transformers", "tokenizers": "tokenizers", "huggingface_hub": "huggingface-hub"}
sudah = [m for m, q in MODUL.items() if m in sys.modules and q in beda]   # versi lama sudah termuat di proses ini
import torch                                                 # pertama kali, sesudah VERSI terpasang
akhir = {p: torch.__version__ if p == "torch" else terpasang(p) for p in VERSI}

# 5) Ringkasan + cek; log versi untuk K4
gpu = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
AKAR_K = Path("/kaggle/working/skripsi_wondr_v4")
(AKAR_K / "log").mkdir(parents=True, exist_ok=True)
(AKAR_K / "log" / "versi_kaggle.json").write_text(json.dumps({"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(),
    "gpu": gpu, "driver_cuda": cuda_drv, "bawaan": bawaan, "terpasang": akhir, "torch": torch.__version__, "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version()}, indent=2))
print(f"{'Package':<17}{'bawaan':<18}{'sekarang':<18}VERSI")
print("\n".join(f"{p:<17}{str(bawaan[p]):<18}{str(akhir[p]):<18}{v}" for p, v in VERSI.items()))
print(f"GPU terlihat: {len(gpu)} · {' · '.join(gpu) or 'tidak ada'} · cuDNN {torch.backends.cudnn.version()}")
for nama, ok, aktual, target in [
        ("platform kaggle", Path("/kaggle/input").is_dir(), "ya" if Path("/kaggle/input").is_dir() else "tidak", "ya"),
        ("jumlah GPU · semuanya T4 (F11)", len(gpu) == 1 and all("T4" in g for g in gpu), f"{len(gpu)} · {gpu}", "1 · T4"),
        ("versi terpasang = VERSI (F7)", akhir == VERSI and torch.__version__ == VERSI["torch"], "sama" if akhir == VERSI else str(
            {p: akhir[p] for p in VERSI if akhir[p] != VERSI[p]}), "sama"),
        ("modul versi lama sudah termuat sebelum pemasangan", not sudah, ", ".join(sudah) or "tidak ada", "tidak ada")]:
    print(f"  {'✅' if ok else '⚠️'} {nama} = {aktual} (target {target})")
if not (len(gpu) == 1 and all("T4" in g for g in gpu) and akhir == VERSI and not sudah):
    raise RuntimeError("K0 gagal: periksa baris ⚠️ di atas (GPU harus satu T4, versi harus = VERSI). Jangan lanjut ke K1.")

## K1 — Data
Paket dibaca dari `/kaggle/input` (zip atau folder hasil ekstraksi), disalin ke `/kaggle/working/skripsi_wondr_v4/`, MD5 dicocokkan, `CFG` dimuat (versi dan revisi harus = K0), lalu *state* sesi sebelumnya dipulihkan.

In [ ]:
# ══ K1 — Data: salin paket ke /kaggle/working, cek MD5 dan VERSI, pulihkan state dari input (tanpa GPU) ══
# Package: json, shutil, zipfile, tempfile, hashlib, pathlib (bawaan Python); lib.bantu, lib.wondr_optim (dari paket)
import json, shutil, zipfile, tempfile, hashlib, sys
from pathlib import Path
INPUT = Path("/kaggle/input")

# 1) Paket bisa berupa folder hasil ekstraksi otomatis Kaggle (memuat manifest_md5.json) atau paket_kaggle.zip
man = sorted(p for p in INPUT.rglob("manifest_md5.json") if (p.parent / "cfg.json").exists())
zips = sorted(INPUT.rglob("paket_kaggle.zip"))
if len({m.read_bytes() for m in man}) > 1:                   # paket lama dan baru sama-sama ada di input
    raise RuntimeError("Ada lebih dari satu versi paket di /kaggle/input: " + ", ".join(str(m.parent) for m in man)
                       + ". Hapus input yang lama (Remove), lalu ulangi K1.")
if man:
    sumber, tmp = man[0].parent, None
elif zips:
    tmp = tempfile.mkdtemp()
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(tmp)
    sumber = Path(tmp)
else:
    raise RuntimeError("Paket tidak ditemukan di /kaggle/input. Tambahkan dataset paket_kaggle lewat Add Input, lalu ulangi K1.")
manifes = json.loads((sumber / "manifest_md5.json").read_text(encoding="utf-8"))
md5 = lambda p: hashlib.md5(Path(p).read_bytes()).hexdigest()
salah = [r for r, h in manifes.items() if not (sumber / r).is_file() or md5(sumber / r) != h]
if salah:
    raise RuntimeError(f"MD5 paket tidak cocok / berkas hilang: {salah}. Unggah ulang paket_kaggle.zip.")
for r in manifes:                                            # salin ke folder kerja yang dapat ditulis; lib berjalan tanpa diubah
    (AKAR_K / r).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(sumber / r, AKAR_K / r)
salah2 = [r for r, h in manifes.items() if md5(AKAR_K / r) != h]

# 2) CFG dari cfg.json; versi dan revisi harus = VERSI dan REVISI di K0
CFG = json.loads((AKAR_K / "cfg.json").read_text(encoding="utf-8"))
if CFG["versi"] != VERSI or CFG["revisi"] != REVISI:
    raise RuntimeError("cfg.json berbeda dengan VERSI/REVISI di K0. Samakan konstanta K0 dengan Tahap 0.1 Colab, lalu ulangi.")
os.chdir(AKAR_K)
if str(AKAR_K) not in sys.path:
    sys.path.insert(0, str(AKAR_K))
from lib.bantu import *
import lib.wondr_latih as wl, lib.wondr_optim as wo, lib.wondr_pso as wp
mulai()
PATH = {"akar": AKAR_K, "hasil": AKAR_K / "hasil", "log": AKAR_K / "log"}

# 3) Pulihkan state RS/PSO dan tahap10_kaggle.json dari semua input (output versi sebelumnya), aturan gabung F10
baris = wo.pulihkan_state(INPUT, PATH["log"], PATH["hasil"], CFG)
judul("K1", "Data dan pemulihan state", masukan=str(zips[0] if not man else sumber), keluaran=str(AKAR_K))
print(f"berkas paket {len(manifes)} · amp {CFG['amp']} · revisi {CFG['revisi'][:12]}… · state dipulihkan: "
      + (" · ".join(f"{b['metode']} {b['seed']} = {b['evaluasi']}" for b in baris) or "tidak ada"))
cek("MD5 setelah disalin · data uji tidak ada di paket · amp sudah diisi", [len(salah2), any("uji" in r for r in manifes),
    CFG["amp"] is not None], [0, False, True])
selesai()

## K2 — Uji kesetaraan Kaggle–Colab
Tahap 10 dijalankan di Kaggle dan dibandingkan dengan hasil Colab: F1 per *epoch*, *epoch* terbaik, dan peluang validasi harus **identik persis**. Bila tidak, sel berhenti (RS dijalankan di Colab). Dilewati bila sudah lulus di sesi sebelumnya. Keluaran: `hasil/tahap10_kaggle.json`.

In [ ]:
# ══ K2 — Uji kesetaraan Kaggle–Colab: Tahap 10 harus identik persis (GPU T4, ±3–8 menit; dilewati bila sudah lulus) ══
# Package: numpy; lib.wondr_latih (fitness); lib.wondr_optim (lingkungan)
import numpy as np
J10K = PATH["hasil"] / "tahap10_kaggle.json"
if J10K.exists() and baca_json(J10K).get("lulus"):
    print("K2 dilewati: uji kesetaraan sudah lulus (hasil/tahap10_kaggle.json dari sesi sebelumnya).")
else:
    mulai()
    wl.pastikan_t4()                                         # F11
    wl.pakai_konfig(CFG)
    d, S = CFG["default"], CFG["seed_fitness"]
    colab, z = baca_json(PATH["hasil"] / "tahap10_default.json"), np.load(PATH["hasil"] / "tahap10_prob_validasi.npz")
    r = wl.fitness(d["lr"], d["wr"], d["wd"], prob=True)
    _, val = wl.data_fitness()
    H = r["hasil"]
    beda = {str(s): float(np.max(np.abs(H[s]["prob"]["validasi"] - z[f"p{s}"]))) for s in S}
    lulus = bool(colab["amp"] == CFG["amp"] and np.array_equal(val.review_id.to_numpy(), z["review_id"])
                 and all(H[s]["f1_per_epoch"] == colab["per_seed"][str(s)]["f1_per_epoch"]
                         and H[s]["epoch_terbaik"] == colab["per_seed"][str(s)]["epoch_terbaik"] and beda[str(s)] == 0 for s in S))
    simpan_json({"lulus": lulus, "amp": CFG["amp"], "f1_per_epoch": {str(s): H[s]["f1_per_epoch"] for s in S},
                 "epoch_terbaik": {str(s): H[s]["epoch_terbaik"] for s in S}, "maks_selisih_peluang": beda,
                 "detik": {str(s): H[s]["detik"] for s in S}, "lingkungan": wo.lingkungan(CFG)}, J10K)
    judul("K2", "Uji kesetaraan Kaggle–Colab", masukan="hasil/tahap10_default.json · hasil/tahap10_prob_validasi.npz (Colab)",
          keluaran="hasil/tahap10_kaggle.json")
    for s in S:
        print(f"seed {s}: F1 per epoch Kaggle [{' · '.join(fmt_id(f, 6) for f in H[s]['f1_per_epoch'])}] · Colab "
              f"[{' · '.join(fmt_id(f, 6) for f in colab['per_seed'][str(s)]['f1_per_epoch'])}] · maks selisih peluang "
              f"{fmt_id(beda[str(s)], 1) if beda[str(s)] == 0 else f'{beda[str(s)]:.1e}'}")
    cek("Tahap 10 Kaggle identik persis dengan Colab (F1 per epoch · epoch terbaik · peluang validasi)", "ya" if lulus else "tidak", "ya")
    selesai()
    if not lulus:
        raise RuntimeError("UJI KESETARAAN GAGAL: RS tidak boleh dijalankan di Kaggle (F10). Jalankan RS di Colab (Tahap 11.1).")

## K3 — Pencarian
`TUGAS` = RS *seed* 1–5, masing-masing 40 kandidat; *state* disimpan setelah setiap kandidat; berhenti rapi 11 jam sejak awal K0.

In [ ]:
# ══ K3 — Pencarian di Kaggle: random search 5 run × 40 kandidat (GPU T4; berhenti rapi sebelum batas sesi) ══
# Package: lib.wondr_latih (fitness); lib.wondr_optim (jalankan_rs); lib.wondr_pso (jalankan)
TUGAS = [("rs", s) for s in CFG["rs_seeds"]]                 # bila RS selesai lebih dulu, boleh ditambah ("pso", 15), ("pso", 14), ... (F10)
if not baca_json(PATH["hasil"] / "tahap10_kaggle.json").get("lulus"):
    raise RuntimeError("Uji kesetaraan (K2) belum lulus; pencarian tidak dijalankan di Kaggle.")
mulai()
wl.pastikan_t4()                                             # F11
wl.pakai_konfig(CFG)
BATAS, p = CFG["kaggle_batas_jam"] * 3600, CFG["pso"]
env, ev = wo.lingkungan(CFG), wo.evaluasi_fitness(wl.fitness, CFG["ruang"])
def cetak_pso(b):
    print(f"PSO seed {b['seed']}: berhenti di t {b['t']} partikel {b['i']}" if b["berhenti"] else
          f"PSO seed {b['seed']} · t {b['t']} · evaluasi {b['evaluasi']}/{p['s'] * (1 + p['G_max'])} · global best {fmt_id(b['gbest'])} · "
          f"{fmt_id(b['menit'], 1)} menit")
for m, s in TUGAS:                                           # semua tugas dibawa ke 40 evaluasi; batas dicek sebelum setiap evaluasi
    if time.time() - T_MULAI >= BATAS:
        print(f"Batas {CFG['kaggle_batas_jam']} jam tercapai; tugas berikutnya dilanjutkan di sesi berikutnya."); break
    if m == "rs":
        wo.jalankan_rs([s], CFG["rs_F"], CFG["ruang"], wl.fitness, env, M_target=CFG["rs_M"], batas_detik=BATAS, t_mulai=T_MULAI)
    else:
        wp.jalankan(wo.path_state("pso", s), ev, p["s"], p["G_max"], p, s, cetak=cetak_pso, batas_detik=BATAS, t_mulai=T_MULAI, env=env)
selesai()

## K4 — Keluaran
Progres per tugas dan zip *state* RS/PSO, `tahap10_kaggle.json`, serta log versi ke `/kaggle/working/keluaran_kaggle_<YYYYmmdd_HHMM>.zip`.

In [ ]:
# ══ K4 — Keluaran: progres per tugas dan zip state untuk diimpor di Colab (Tahap 11.2) ══
# Package: time, zipfile (bawaan Python)
import time, zipfile
mulai()
p = CFG["pso"]
N = {"rs": CFG["rs_M"], "pso": p["s"] * (1 + p["G_max"])}

# 1) Progres setiap state yang ada di log/
prog = []
for m in ("rs", "pso"):
    for q in sorted(PATH["log"].glob(f"{m}_state_seed*.json")):
        st = baca_json(q)
        prog.append((q.name, m, len(st["hasil" if m == "rs" else "log"])))

# 2) Zip state RS dan PSO, hasil uji kesetaraan, dan log versi ke /kaggle/working (tab Output)
ZIP = Path("/kaggle/working") / f"keluaran_kaggle_{time.strftime('%Y%m%d_%H%M')}.zip"
isi = [q for q in sorted(PATH["log"].glob("*_state_seed*.json"))] + [q for q in (PATH["hasil"] / "tahap10_kaggle.json",
       PATH["log"] / "versi_kaggle.json") if q.exists()]
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for q in isi:
        z.write(q, q.relative_to(AKAR_K))
judul("K4", "Keluaran Kaggle", masukan="log/*_state_seed*.json · hasil/tahap10_kaggle.json", keluaran=str(ZIP))
for nama, m, n in prog:
    print(f"  {nama:<24} {n:>2}/{N[m]} evaluasi")
print("Langkah: unduh zip ini dari tab Output, unggah ke MyDrive/skripsi_wondr_v4/kaggle/, lalu jalankan Tahap 11.2 di Colab.")
cek("berkas di zip · tugas RS lengkap", [len(isi), sum(n >= N[m] for _, m, n in prog if m == "rs")],
    [len(prog) + 2, len(CFG["rs_seeds"])])
selesai()